In [20]:
import itertools
states=[]
mdp={}

actions={1:"Right",-1:"Left",-4:"Up",4:"Down"}

bla=[i+1 for i in range(16)]
tabla=tuple(bla)

def getitdone(p, q, cfg, transitions, r):
    abc=cfg.copy()
    abc[p]=abc[q]
    abc[q]=16
    if abc[:(r+1)*4]==bla[:(r+1)*4]:
        transitions[actions[q-p]]=[tuple(abc),75+(r+1)*25]
    else:
        noc=0
        for i in range((r+1)*4):
            if abc[i]==bla[i]:
                noc+=1
        transitions[actions[q-p]]=[tuple(abc),-50+2*noc]

def all_configs():
    for positions in itertools.combinations(range(16), 5):
        for values in itertools.permutations([1, 2, 3, 4, 16]):
            lst = [0] * 16
            for pos, val in zip(positions, values):
                lst[pos] = val
            yield lst
for cfg in all_configs():
    states.append(tuple(cfg))
    x=-1
    for i in range(16):
        if cfg[i]==16:
            x=i
            break
    transitions={}
    for i in actions:
        transitions[actions[i]]=[tuple(cfg),-1000]
    if x%4!=3:
        getitdone(x,x+1,cfg,transitions,0)
    if x%4!=0:
        getitdone(x,x-1,cfg,transitions,0)
    if x>3:
        getitdone(x,x-4,cfg,transitions,0)
    if x<12:
        getitdone(x,x+4,cfg,transitions,0)
    mdp[tuple(cfg)]=transitions

In [21]:
mdp1={}

def all_configs1():
    for positions in itertools.combinations(range(4,16), 5):
        for values in itertools.permutations([5, 6, 7, 8, 16]):
            lst = [0] * 16
            lst[:4]=bla[:4]
            for pos, val in zip(positions, values):
                lst[pos] = val
            yield lst
for cfg in all_configs1():
    states.append(tuple(cfg))
    x=-1
    for i in range(4,16):
        if cfg[i]==16:
            x=i
            break
    transitions={}
    for i in actions:
        transitions[actions[i]]=[tuple(cfg),-100]
    if x%4!=3:
        getitdone(x,x+1,cfg,transitions,1)
    if x%4!=0:
        getitdone(x,x-1,cfg,transitions,1)
    if x>7:
        getitdone(x,x-4,cfg,transitions,1)
    if x<12:
        getitdone(x,x+4,cfg,transitions,1)
    mdp1[tuple(cfg)]=transitions

In [22]:
mdp2={}

def all_configs2():
    for positions in itertools.combinations(range(8,16), 8):
        for values in itertools.permutations([9,10,11,12,13,14,15,16]):
            lst = [0] * 16
            lst[:8]=bla[:8]
            for pos, val in zip(positions, values):
                lst[pos] = val
            yield lst
for cfg in all_configs2():
    states.append(tuple(cfg))
    x=-1
    for i in range(4,16):
        if cfg[i]==16:
            x=i
            break
    transitions={}
    for i in actions:
        transitions[actions[i]]=[tuple(cfg),-100]
    if x%4!=3:
        getitdone(x,x+1,cfg,transitions,3)
    if x%4!=0:
        getitdone(x,x-1,cfg,transitions,3)
    if x>11:
        getitdone(x,x-4,cfg,transitions,3)
    if x<12:
        getitdone(x,x+4,cfg,transitions,3)
    mdp2[tuple(cfg)]=transitions

In [23]:
v={i:0.0 for i in states}
v[tabla]=1.0
theta=1e-2
gamma=0.9

In [25]:
while True:
    delta=0
    vnew=v.copy()
    for s in states:
        p=s.count(0)
        if s==tabla:
            continue
        elif p==0:
            best=float('-inf')
            for i in actions:
                reward=mdp2[s][actions[i]][1]
                next=mdp2[s][actions[i]][0]
                best=max(best,(reward+v[next]*gamma))
            delta=max(delta,abs(v[s]-best))   
            vnew[s]=best
        elif p==7:
            best=float('-inf')
            for i in actions:
                reward=mdp1[s][actions[i]][1]
                next=mdp1[s][actions[i]][0]
                best=max(best,(reward+v[next]*gamma))
            delta=max(delta,abs(v[s]-best))   
            vnew[s]=best
        else:
            best=float('-inf')
            for i in actions:
                reward=mdp[s][actions[i]][1]
                next=mdp[s][actions[i]][0]
                best=max(best,(reward+v[next]*gamma))
            delta=max(delta,abs(v[s]-best))   
            vnew[s]=best
    v=vnew
    if delta<theta:
        break

In [26]:
policy={}
for s in states:
    p=s.count(0)
    best=float('-inf')
    besta=None
    if s==tabla:
        continue
    elif p==0:
        for i in actions:
            reward=mdp2[tuple(s)][actions[i]][1]
            next=mdp2[tuple(s)][actions[i]][0]
            ba=reward+v[next]*gamma
            if ba>best:
                best=ba
                besta=actions[i]
        policy[s]=besta
    elif p==7:
        for i in actions:
            reward=mdp1[tuple(s)][actions[i]][1]
            next=mdp1[tuple(s)][actions[i]][0]
            ba=reward+v[next]*gamma
            if ba>best:
                best=ba
                besta=actions[i]
        policy[s]=besta
    else:
        for i in actions:
            reward=mdp[tuple(s)][actions[i]][1]
            next=mdp[tuple(s)][actions[i]][0]
            ba=reward+v[next]*gamma
            if ba>best:
                best=ba
                besta=actions[i]
        policy[s]=besta
    